# Preference Tuning (DPO) on GEAP

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package rather than re-implementing logic. See
[`docs/notes/tuning-apis.md`](../docs/notes/tuning-apis.md) for the API details.

Where [SFT](01_sft.ipynb) taught the model *what* a ticket is about, DPO teaches
it *how* to reply. Each record pairs a customer message with a **preferred** and
a **dispreferred** completion (`score` 1 vs 0); the model learns to widen the
likelihood gap between them. It uses the *same* `client.tunings.tune(...)` call
as SFT, with `method="PREFERENCE_TUNING"` and a `beta` coefficient.

> **Best practice:** SFT on the preferred responses first, then continuous-tune
> from that checkpoint with DPO. This demo runs DPO directly on the base model
> to stay self-contained.

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place before running the tune/eval cells.

In [ ]:
from geap_tuning.config import genai_client, load_config

cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the preference dataset

Deterministic train/val/test splits of `(message, preferred, dispreferred)`
triples.

In [ ]:
from geap_tuning.preference.data import build_preference_dataset

paths = build_preference_dataset("../datasets/preference_support_style")
paths

In [ ]:
import json
from pathlib import Path

first = Path(paths["train"]).read_text(encoding="utf-8").splitlines()[0]
print(json.dumps(json.loads(first), indent=2, ensure_ascii=False))

## 2. Stage the splits to GCS

In [ ]:
from geap_tuning.gcs import upload_file

train_uri = upload_file(paths["train"], f"{cfg.bucket}/preference_support_style/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/preference_support_style/val.jsonl")
train_uri, val_uri

## 3. Launch the DPO job and wait

Reuse an existing job with the same display name if one exists (cost control).
`beta` (recommended 0.01-0.5) controls how closely the tuned model stays to the
base; lower means more aggressive updates toward the preferred response.

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.preference.tune import launch_preference_job

DISPLAY_NAME = "geap-dpo-support-style"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_preference_job(
        client, train_uri=train_uri, val_uri=val_uri, display_name=DISPLAY_NAME, beta=0.1
    )
job = wait_for_tuning_job(client, job.name)
job.state

## 4. Evaluate the tuned endpoint

DPO has no single gold answer, so we measure an autorater **win-rate**: how often
the tuned reply beats the dispreferred reference in a blind A/B judgment (a base
Gemini model is the judge).

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import tuned_endpoint
from geap_tuning.preference.data import SUPPORT_REPLIES, build_preference_records, split_dataset
from geap_tuning.preference.evaluate import run_preference_eval

JUDGE_MODEL = "gemini-2.5-flash"
JUDGE_PROMPT = (
    "You are judging two customer-support replies to the same message. Pick the "
    "reply that is warmer, more concise, and more helpful. Answer with only 'A' "
    "or 'B'.\n\nCustomer message: {user}\n\nReply A: {a}\n\nReply B: {b}\n\nBetter reply:"
)


def judge_fn(user_text: str, cand_a: str, cand_b: str) -> str:
    verdict = generate(client, JUDGE_MODEL, JUDGE_PROMPT.format(user=user_text, a=cand_a, b=cand_b))
    return verdict[:1].upper()


endpoint = tuned_endpoint(job)
_, _, test_triples = split_dataset(SUPPORT_REPLIES)
metrics = run_preference_eval(
    build_preference_records(test_triples),
    generate_fn=lambda user_text: generate(client, endpoint, user_text),
    judge_fn=judge_fn,
)
print(f"Tuned-vs-dispreferred win rate: {metrics['win_rate']:.3f} (n={metrics['n']})")

## Next steps

- **RLFT** — a `references` dataset plus a reward function over REST `v1beta1`
  (`notebooks/03_rlft.ipynb`, future).